In [0]:
# python imports
from pyspark.sql import functions as F
import scipy.stats as stats
import numpy as np
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import plotly.express as px
import scipy.sparse as sp



In [0]:
# some sample product maps

import pandas as pd

# Steel/iron -> automotive chain (linear subset)
product_map_steel = pd.DataFrame([
    ("8703", "8708", 1.00),   # vehicle <- parts/accessories
    ("8708", "7210", 1.00),   # parts <- coated/plated flat steel
    ("7210", "7209", 1.00),   # coated flat steel <- cold-rolled flat steel
    ("7209", "7208", 1.00),   # cold-rolled flat steel <- hot-rolled flat steel
    ("7208", "7207", 1.00),   # hot-rolled flat steel <- semi-finished steel
    ("7207", "2601", 1.00),   # semi-finished steel <- iron ore
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_steel = ["8703", "8708", "7210", "7209", "7208", "7207", "2601"]


# Aluminum/bauxite chain
product_map_al = pd.DataFrame([
    ("8703", "8707", 1.00),   # vehicle <- body
    ("8707", "7616", 1.00),   # body <- aluminum articles
    ("7616", "7601", 1.00),   # aluminum articles <- unwrought aluminum
    ("7601", "2606", 1.00),   # unwrought aluminum <- bauxite
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_al = ["8703", "8707", "7616", "7601", "2606"]


# Copper chain
product_map_cu = pd.DataFrame([
    ("8703", "8501", 1.00),   # vehicle <- electric motor
    ("8501", "7408", 1.00),   # motor <- copper wire
    ("7408", "7403", 1.00),   # copper wire <- refined copper
    ("7403", "2603", 1.00),   # refined copper <- copper ore
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_cu = ["8703", "8501", "7408", "7403", "2603"]

product_map_semi = pd.DataFrame([
    ("8471", "8542", 1.00),   # computers <- integrated circuits
    ("8542", "3818", 1.00),   # integrated circuits <- doped silicon wafers/chemical elements for electronics
    ("3818", "2804", 1.00),   # doped silicon wafers <- silicon
    ("2804", "2811", 1.00),   # silicon <- other inorganic acids/compounds (silica precursor proxy)
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_semi = ["8471", "8542", "3818", "2804", "2811"]

product_map_textile = pd.DataFrame([
    ("6109", "5208", 1.00),   # t-shirts (cotton) <- woven cotton fabric
    ("5208", "5205", 1.00),   # woven cotton fabric <- cotton yarn
    ("5205", "5201", 1.00),   # cotton yarn <- raw cotton, not carded/combed
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_textile = ["6109", "5208", "5205", "5201"]

product_map_cocoa = pd.DataFrame([
    ("1806", "1803", 1.00),   # chocolate <- cocoa paste
    ("1803", "1801", 1.00),   # cocoa paste <- cocoa beans, whole or broken, raw or roasted
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_cocoa = ["1806", "1803", "1801"]

product_map_semiconductor = pd.DataFrame([
    ("8542", "3818", 1.00),   # integrated circuits <- doped semiconductor wafers
    ("3818", "2804", 1.00),   # doped wafers <- silicon (polysilicon feedstock)
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_semiconductor = ["8542", "3818", "2804"]

In [0]:
def load_global_net_exports(hs_codes):
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    query = f"""
        SELECT * 
        FROM silver_country_hs_code_hyperparamers_2
        WHERE hs_code IN ({hs_codes_str})
        AND a IS NOT NULL
    """
    share = spark.sql(query).toPandas()
    share['net_exports'] = share['exports']-share['imports']
    share = share[share['net_exports']>0]
    share = share[['hs_code', 'country_code', 'country_name', 'exports', 'imports', 'net_exports']]

    global_share = share.groupby(['hs_code']).net_exports.sum().reset_index()
    share = pd.merge(share, global_share, on = 'hs_code', suffixes=('', '_global'))
    share['global_share'] = share['net_exports']/share['net_exports_global']
    return share


In [0]:
   
    hs_codes=hs_chain_steel
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    query = f"""
        SELECT * 
        FROM silver_country_hs_code_hyperparamers_4
        WHERE hs_code IN ({hs_codes_str})
        AND a IS NOT NULL
    """
    share = spark.sql(query).toPandas()

In [0]:
share[share['country_code']=='KR']

In [0]:
def load_domestic_share_data(hs_codes):
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    query = f"""
        SELECT * 
        FROM silver_country_hs_code_hyperparamers_4
        WHERE hs_code IN ({hs_codes_str})
        AND a IS NOT NULL
    """
    share = spark.sql(query).toPandas()
    return share


def generate_domestic_production_share(domestic_share_data, seed):
    share = domestic_share_data.copy()
    share['alpha_max']=(share['imports']/share['exports']).fillna(0).clip(upper=1)
    rng = np.random.default_rng(seed)
    share['alpha'] = rng.uniform(
        0,
        share['alpha_max'].to_numpy()
    )
    share['domestic_share'] = (1-share['alpha'])

    share = domestic_share_data.copy()
    # Define alpha_floor
    x = share['exports'].replace(0, np.nan)
    m = share['imports'].fillna(0)
    share['alpha_min'] = np.maximum(0, 1 - m / x)
    share['alpha_min'] = share['alpha_min'].fillna(0).clip(0, 1)

    # Generate alphas
    rng = np.random.default_rng(seed)
    share['alpha_raw'] = rng.beta(
        share['a'].to_numpy(),
        share['b'].to_numpy()
    )
    share['alpha'] = (
        share['alpha_min']
        + (1 - share['alpha_min']) * share['alpha_raw']
    )

    share['domestic_share'] = share['alpha']
    share = share[['hs_code', 'country_code', 'domestic_share']]

    # Starting to build out some over rides when the input data is corrupted
    ae_domestic_hs4_exceptions = [
    "2709",  # Crude petroleum oils (ADNOC)
    "2710",  # Refined petroleum products
    "2711",  # Petroleum gases / natural gas, incl. LNG
    "7601",  # Unwrought aluminium — Emirates Global Aluminium (EGA), one of the world's largest smelters
    "3901",  # Polymers of ethylene (polyethylene) — Borouge, ~6.6M tonnes/year capacity by 2028
    "3902",  # Polymers of propylene (polypropylene) — Borouge
    "3102",  # Nitrogenous fertilizers (urea) — Fertiglobe, one of the world's largest producers
    "2814",  # Ammonia — Fertiglobe's other core product, often coded separately from urea
    "7213",  # Bars and rods, hot-rolled, of iron/steel — EMSTEEL (rebar, wire rod)
    "7214",  # Other bars/rods of iron/steel, not further worked — EMSTEEL
    "2523",  # Cement — EMSTEEL/Arkan; genuine domestic capacity, but check current export volumes since global significance here is more regional (GCC) than the others
    ]
    mask = (share['country_code'] == 'AE') & (~share['hs_code'].isin(ae_domestic_hs4_exceptions))
    share.loc[mask, 'domestic_share'] = 0

    return share

def load_import_shares(hs_codes):
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    # Pull only the records you need directly from the cluster database
    query = f"""
        SELECT * 
        FROM silver_country_imports 
        WHERE hs_code IN ({hs_codes_str})
    """
    imports = spark.sql(query).toPandas()
    imports = imports[['hs_code', 'country_code', 'source_country_code', 'import_share']]
    return imports

In [0]:
def generate_transition_matrices(product_map, domestic_share_data, import_share_data, seed = 42):  

    hs_codes = set(product_map['target_hs_code']).union(set(product_map['source_hs_code']))

    # Generate within-country & hs-code terminal transitions
    domestic_share = generate_domestic_production_share(domestic_share_data, seed)

    domestic_share['source_hs_code']=domestic_share['hs_code']
    domestic_share['target_hs_code']=domestic_share['hs_code']
    domestic_share['source_country_code']=domestic_share['country_code']
    domestic_share['target_country_code']=domestic_share['country_code']
    import_share = domestic_share.copy()
    domestic_share['source_country_terminal']='production'
    domestic_share['target_country_terminal']='supply'
    import_share['source_country_terminal']='import'
    import_share['target_country_terminal']='supply'
    domestic_share['weight']=domestic_share['domestic_share']
    import_share['weight']=1-import_share['domestic_share']

    domestic_share = domestic_share.drop(['hs_code','country_code', 'domestic_share'], axis = 1)
    import_share = import_share.drop(['hs_code','country_code', 'domestic_share'], axis = 1)

    # Generate across-country trade transitions
    imports = import_share_data.copy()
    imports['source_hs_code']=imports['hs_code']
    imports['target_hs_code']=imports['hs_code']
    imports['target_country_code']=imports['country_code']
    imports['source_country_terminal']='supply'
    imports['target_country_terminal']='import'
    imports['weight']=imports['import_share']
    imports = imports.drop(['hs_code','country_code', 'import_share'], axis = 1)
    imports= imports[imports['weight'].notnull()]

    # Generate absorbing edges
    absorbing_edges = pd.DataFrame(hs_codes, columns = ['source_hs_code'])
    absorbing_edges['target_hs_code']=absorbing_edges['source_hs_code']
    absorbing_edges['target_country_terminal']='production'
    absorbing_edges['source_country_terminal']='production'
    country_codes = set(imports['target_country_code']).union(set(imports['source_country_code']))

    # Replicate every product_map row for every country
    absorbing_edges = (
        absorbing_edges
        .merge(
            pd.DataFrame({'source_country_code': list(country_codes)}),
            how='cross'
        )
    )

    absorbing_edges['target_country_code'] = absorbing_edges['source_country_code']
    absorbing_edges['weight']=1
    global_transition_matrix = pd.concat([absorbing_edges, import_share, domestic_share, imports])

    # Assign indices to tuples
    cols = [c for c in global_transition_matrix.columns if c != 'weight']
    global_transition_matrix['source_tuple'] = global_transition_matrix[['source_hs_code', 'source_country_code', 'source_country_terminal']].apply(tuple, axis=1)
    global_transition_matrix['target_tuple'] = global_transition_matrix[['target_hs_code', 'target_country_code', 'target_country_terminal']].apply(tuple, axis=1)

    tuples = pd.concat([
        global_transition_matrix['source_tuple'],
        global_transition_matrix['target_tuple']
    ]).drop_duplicates().reset_index(drop=True)

    tuples = tuples.to_frame(name='tuple')
    tuples = tuples.reset_index()

    # Create dictionaries
    index_to_tuple = tuples['tuple'].to_dict()

    tuple_to_index = {
        value: key
        for key, value in index_to_tuple.items()
    }

    Q_matrix = pd.concat([import_share, imports])
    R_matrix = domestic_share

    # Assign indices to matrices
    for df in (Q_matrix, R_matrix, global_transition_matrix):
        df['source_tuple'] = df[['source_hs_code', 'source_country_code', 'source_country_terminal']].apply(tuple, axis=1)
        df['target_tuple'] = df[['target_hs_code', 'target_country_code', 'target_country_terminal']].apply(tuple, axis=1)
        df['col'] = df['source_tuple'].map(tuple_to_index)
        df['row'] = df['target_tuple'].map(tuple_to_index)

    n = len(tuple_to_index)

    # Build sparse matrices
    Q_full = sp.coo_matrix((Q_matrix['weight'], (Q_matrix['row'], Q_matrix['col'])), shape=(n, n)).tocsr()
    R_full = sp.coo_matrix((R_matrix['weight'], (R_matrix['row'], R_matrix['col'])), shape=(n, n)).tocsr()
    T_full = sp.coo_matrix((global_transition_matrix['weight'], (global_transition_matrix['row'], global_transition_matrix['col'])), shape=(n, n)).tocsr()

    # Reduce index space for sub matrices
    tuples['is_terminal'] = tuples['tuple'].apply(lambda t: t[2] == 'production')
    transient_idx = tuples.index[~tuples['is_terminal']].to_numpy()
    absorbing_idx = tuples.index[tuples['is_terminal']].to_numpy()
    Q = Q_full[np.ix_(transient_idx, transient_idx)]
    R = R_full[np.ix_(transient_idx, absorbing_idx)]

    # Solve for fundamental matrix
    I = sp.identity(Q.shape[0], format='csc')
    A = (I - Q).tocsc()

    F = sp.linalg.inv(A)   

    # Calculate absorbtion matrix
    B = F @ R.toarray()

    def sparse_to_long(matrix, row_idx_map, col_idx_map):
        coo = sp.coo_matrix(matrix)
        df = pd.DataFrame({
            'source_tuple': [index_to_tuple[col_idx_map[c]] for c in coo.col],
            'target_tuple': [index_to_tuple[row_idx_map[r]] for r in coo.row],
            'weight': coo.data,
        })
        df[['source_hs_code', 'source_country_code', 'source_country_terminal']] = pd.DataFrame(df['source_tuple'].tolist(), index=df.index)
        df[['target_hs_code', 'target_country_code', 'target_country_terminal']] = pd.DataFrame(df['target_tuple'].tolist(), index=df.index)
        return df

    # Back to pandas
    full_idx = np.arange(n)
    T_long = sparse_to_long(T_full, full_idx, full_idx)
    F_long = sparse_to_long(F, transient_idx, transient_idx)
    B_long = sparse_to_long(B, transient_idx, absorbing_idx)

    # Caculate reduced transition matrix and add product transitions
    min_threshold = 0.001
    country_transitions = B_long[(B_long['target_country_terminal']=='supply') & (B_long['weight']>=min_threshold)]
    # country_transitions = country_transitions.drop(['source_country_terminal', 'target_country_terminal', 'source_tuple', 'target_tuple'], axis = 1)
    country_codes = set(country_transitions['target_country_code']).union(set(country_transitions['source_country_code']))
    product_transitions = product_map.copy()

    # Replicate every product_map row for every country
    product_transitions = (
        product_transitions
        .merge(
            pd.DataFrame({'source_country_code': list(country_codes)}),
            how='cross'
        )
    )
    product_transitions['target_country_code']=product_transitions['source_country_code']
    # production_transitions = (country_transitions[(country_transitions['target_country_code']==country_transitions['source_country_code']) & (country_transitions['source_hs_code']==country_transitions['target_hs_code'])].sort_values('weight', ascending = False))
    # production_transitions = pd.merge(production_transitions, product_map, how = 'inner', on='target_hs_code')
    # production_transitions['weight']= production_transitions['weight_x']*production_transitions['weight_y']
    # production_transitions = production_transitions.drop(['source_tuple', 'target_tuple','weight_x', 'weight_y', 'source_hs_code_x'], axis = 1).rename(columns={'source_hs_code_y': 'source_hs_code'})
    product_transitions['source_country_terminal'] = 'supply'
    product_transitions['target_country_terminal'] = 'production'


    reduced_transition_matrix = pd.concat([country_transitions, product_transitions])
    reduced_transition_matrix['source_tuple'] = reduced_transition_matrix[['source_hs_code', 'source_country_code', 'source_country_terminal']].apply(tuple, axis=1)
    reduced_transition_matrix['target_tuple'] = reduced_transition_matrix[['target_hs_code', 'target_country_code', 'target_country_terminal']].apply(tuple, axis=1)

    return global_transition_matrix, F_long, reduced_transition_matrix


In [0]:
def generate_start_vector(root_hs_code, start_country, number_of_transitions, tuple_to_index):
    start_node_tuple = (root_hs_code, start_country, 'supply')
    start_node = tuple_to_index.get(start_node_tuple)
    start_vector = csr_matrix(([1.0], ([0], [start_node])), shape=(1, number_of_transitions))
    return start_vector

def prepare_trace_inputs(input_matrix, start_hs_code, start_country, skip_start = True):
    transition_matrix = input_matrix.copy()
    if skip_start:
        mask = (
            (transition_matrix['target_country_code'] == start_country) &
            (transition_matrix['target_hs_code'] == start_hs_code) &
            (transition_matrix['target_country_terminal'] == 'supply') &
            (transition_matrix['source_country_code'] == start_country)
        )
        transition_matrix.loc[mask, 'weight'] = 0

        group_mask = (
            (transition_matrix['target_country_code'] == start_country) &
            (transition_matrix['target_hs_code'] == start_hs_code) &
            (transition_matrix['target_country_terminal'] == 'supply')
        )
        group_sum = transition_matrix.loc[group_mask, 'weight'].sum()
        transition_matrix.loc[group_mask, 'weight'] /= group_sum        

    tuples = pd.concat([
        transition_matrix['source_tuple'],
        transition_matrix['target_tuple']
    ]).drop_duplicates().reset_index(drop=True)

    tuples = tuples.to_frame(name='tuple')
    tuples = tuples.reset_index()

    # Create dictionaries
    index_to_tuple = tuples['tuple'].to_dict()

    tuple_to_index = {
        value: key
        for key, value in index_to_tuple.items()
    }

    # Convert to sparse matrix:
    n_nodes = len(index_to_tuple)

    # Reduce to transition matrix
    transition_matrix['source_node']= transition_matrix['source_tuple'].map(tuple_to_index)
    transition_matrix['target_node']= transition_matrix['target_tuple'].map(tuple_to_index)
    transition_matrix['source_node']= transition_matrix['source_node'].astype(int)
    transition_matrix['target_node']= transition_matrix['target_node'].astype(int)
    transition_matrix = transition_matrix[['source_node', 'target_node', 'weight']]

    T = csr_matrix(
        (
            transition_matrix['weight'].values,
            (
                transition_matrix['target_node'].values,
                transition_matrix['source_node'].values
            )
        ),
        shape=(n_nodes, n_nodes)
    )
    start_vector = generate_start_vector(start_hs_code, start_country, T.shape[0], tuple_to_index)

    return T, start_vector, index_to_tuple, tuple_to_index

In [0]:
# Iterate 
import scipy.sparse as sp

def supply_network_trace(start_vector, transition_matrix, n_steps):
    vector = start_vector.copy()
    cumulative_vector = vector.copy()
    cumulative_edges = sp.csr_matrix(transition_matrix.shape)

    for i in range(n_steps):
        v_flat = np.asarray(vector.todense()).ravel() if sp.issparse(vector) else np.asarray(vector).ravel()
        step_edges = sp.diags(v_flat) @ transition_matrix
        cumulative_edges += step_edges

        vector = vector @ transition_matrix
        cumulative_vector += vector

        print(f"Step {i + 1}: {vector.sum()}")

    # one final edge-recording pass so the LAST step's arrivals are captured
    v_flat = np.asarray(vector.todense()).ravel() if sp.issparse(vector) else np.asarray(vector).ravel()
    if v_flat.sum() > 1e-9:
        cumulative_edges += sp.diags(v_flat) @ transition_matrix
    return cumulative_vector, cumulative_edges

def extract_vector(vector, index_to_tuple):
    series = pd.Series(vector.toarray().ravel())
    long_df = pd.DataFrame(
        [index_to_tuple[i] for i in series.index],
        columns=["hs_code", "country_code", "terminal"]
    )
    long_df["weight"] = series.values
    return long_df


def extract_edges(edges, index_to_tuple, min_value=1e-9):
    coo = edges.tocoo()
    rows = []
    for i, j, v in zip(coo.row, coo.col, coo.data):
        if v < min_value:
            continue
        tgt = index_to_tuple[i]   # row = target / downstream / consumer
        src = index_to_tuple[j]   # col = source / upstream / supplier
        rows.append({
            "source_hs": src[0], "source_country": src[1], "source_terminal": src[2],
            "target_hs": tgt[0], "target_country": tgt[1], "target_terminal": tgt[2],
            "weight": v,
        })
    return pd.DataFrame(rows)


In [0]:
import random


def network_trace_iterator(product_map, start_country, start_hs, replicates):
    sim_vectors = pd.DataFrame()
    sim_edges = pd.DataFrame()
    hs_codes = set(product_map['target_hs_code']).union(set(product_map['source_hs_code']))
    domestic_share_data = load_domestic_share_data(hs_codes)
    import_share_data = load_import_shares(hs_codes)

    for i in range(replicates):
        seed = random.randint(0, 2**32 - 1)
        print(f'Iteration {i} with random seed {seed}')
        global_transition_matrix, F_long, reduced_transition_matrix = generate_transition_matrices(product_map, domestic_share_data, import_share_data, seed = seed)
        transition_matrix, start_vector, index_to_tuple, tuple_to_index = prepare_trace_inputs(reduced_transition_matrix, start_hs, start_country, True)
        vector, edges = supply_network_trace(start_vector, transition_matrix, 20)
        vector_df = extract_vector(vector, index_to_tuple)
        edges_df = extract_edges(edges, index_to_tuple)
        vector_df['replicate'] = i
        edges_df['replicate'] = i
        sim_vectors = pd.concat([sim_vectors, vector_df])
        sim_edges = pd.concat([sim_edges, edges_df])
    
    origin_vectors = sim_vectors[sim_vectors['terminal']=='production']

    group_cols = [c for c in origin_vectors.columns if c not in ['weight', 'replicate']]

    all_replicates = sim_vectors['replicate'].unique()

    # Get every distinct combination of group_cols that appears anywhere in origin_vectors
    combos = origin_vectors[group_cols].drop_duplicates()

    # Cross-join combos with every replicate, so every (combo, replicate) pair exists exactly once
    full_index = combos.merge(pd.DataFrame({'replicate': all_replicates}), how='cross')

    # Left-join actual weights onto the full index; missing rows become NaN, then fill with 0
    origin_vectors_full = full_index.merge(
        origin_vectors[group_cols + ['replicate', 'weight']],
        on=group_cols + ['replicate'],
        how='left'
    )
    origin_vectors_full['weight'] = origin_vectors_full['weight'].fillna(0)

    average_origin_vector = origin_vectors_full.groupby(group_cols).agg({'weight': 'mean'}).reset_index()
    median_origin_vector = origin_vectors_full.groupby(group_cols).agg({'weight': 'median'}).reset_index()
    variation_origin_vector = origin_vectors_full.groupby(group_cols).agg({'weight': 'std'}).reset_index()
    origin_vectors_full = (
        average_origin_vector
        .merge(variation_origin_vector, on=['hs_code', 'country_code', 'terminal'])
        .merge(median_origin_vector, on=['hs_code', 'country_code', 'terminal'])
        .rename(columns={
            'weight_x': 'average',
            'weight_y': 'standard_deviation',
            'weight': 'median',
        })
    )
    
    origin_vectors_full['max_std_given_mean'] = np.sqrt(
    origin_vectors_full['average'] * (1 - origin_vectors_full['average'])
    )
    eps = 1E-6
    origin_vectors_full['certainty'] = np.maximum(
        0,
        1 - origin_vectors_full['standard_deviation'] / (origin_vectors_full['average'] + eps)
    )
    # origin_vectors_full['certainty']= np.maximum(0, 1 - origin_vectors_full['standard_deviation'] * np.sqrt(12))  

    scale = origin_vectors_full.groupby('hs_code')['average'].transform('sum')

    origin_vectors_full['scaled_average'] = origin_vectors_full['average'] / scale
    origin_vectors_full['scaled_standard_deviation'] = origin_vectors_full['standard_deviation'] / scale
    origin_vectors_full['scaled_median'] = origin_vectors_full['median'] / scale
    origin_vectors_full['scaled_certainty']= origin_vectors_full['certainty'] 
    return sim_vectors, sim_edges, origin_vectors_full



In [0]:

def plot_prioritisation_matrix(origins, threshold, measure = 'scaled_average'):
    df = origins[origins['average']>threshold].copy()

    df['hs_code_label'] = df['hs_code'].astype(str).map(HS_LABELS)

    # fall back to raw code if any hs_code isn't in the dictionary
    df['hs_code_label'] = df['hs_code_label'].fillna(origins['hs_code'].astype(str))

    fig = px.scatter(
        df,
        x=measure,
        y='scaled_certainty',
        color='hs_code_label',
        hover_data=['hs_code', 'country_code'],
        labels={
            'average': 'Estimated Exposure',
            'certainty': 'Estimate Confidence',
            'hs_code_label': 'HS Code',
        },
    )

    fig.update_traces(
        marker=dict(size=8),
        hovertemplate=(
            "Country: %{customdata[1]}<br>"
            "Estimated Exposure: %{x:.1%}<br>"
            "Estimate Confidence: %{y:.1%}"
            "<extra></extra>"
        ),
    )

    X_MIN, X_MAX = 0, 1
    Y_MIN, Y_MAX = 0, 1

    X_SPLIT = 0.2
    Y_SPLIT = 0.75

    fig.add_vline(x=X_SPLIT, line_dash="dash", line_color="gray")
    fig.add_hline(y=Y_SPLIT, line_dash="dash", line_color="gray")

    quadrant_labels = [
        dict(x=(X_SPLIT + X_MAX) / 2, y=(Y_SPLIT + Y_MAX) / 2,
            text="High priority<br>(material & confident)"),
        dict(x=(X_MIN + X_SPLIT) / 2, y=(Y_SPLIT + Y_MAX) / 2,
            text="Low priority<br>(minor & confident)"),
        dict(x=(X_SPLIT + X_MAX) / 2, y=(Y_MIN + Y_SPLIT) / 2,
            text="Investigate<br>(material & uncertain)"),
        dict(x=(X_MIN + X_SPLIT) / 2, y=(Y_MIN + Y_SPLIT) / 2,
            text="Ignore<br>(minor & uncertain)"),
    ]

    for q in quadrant_labels:
        fig.add_annotation(
            x=q["x"], y=q["y"],
            text=q["text"],
            showarrow=False,
            font=dict(size=13, color="gray"),
            opacity=0.6,
        )

    fig.update_layout(
        height=800,
        width=1000,
        title=dict(
            text="Supply Network Risk Assessment",
            x=0.5,
            xanchor='center',
        ),
        xaxis=dict(
            tickformat='.0%',
            range=[X_MIN, X_MAX],
        ),
        yaxis=dict(
            tickformat='.0%',
            range=[Y_MIN, Y_MAX],
        ),
        legend_title_text='Components',
    )

    fig.show()



In [0]:

def plot_exposure_distribution(sim_vectors, hs_code, country):
    df = sim_vectors[(sim_vectors['hs_code']==hs_code) & (sim_vectors['country_code']==country) & (sim_vectors['terminal']=='production')].copy()
    hs_code_val = str(df['hs_code'].iloc[0])
    country_val = df['country_code'].iloc[0]
    hs_label = hs_code_labels.get(hs_code_val, hs_code_val)

    mean_val = df['weight'].mean()
    median_val = df['weight'].median()

    fig = px.histogram(
        df,
        x='weight',
        labels={'weight': 'Exposure'},
        title=f'{country_val} – {hs_label} — Exposure Distribution Across Replicates',
    )
    n_bins = 50
    fig.update_traces(
        xbins=dict(start=0, end=1, size=1 / n_bins)
    )

    fig.update_layout(
        height=500,
        width=700,
        xaxis=dict(
            tickformat='.0%',
            range=[0, 1],
        ),
        bargap=0.05,
    )

    fig.show()

# hs_code = '87'
# country = 'CL'
# plot_exposure_distribution(sim_vectors, hs_code, country)
# hs_code = '2603'
# country = 'BR'
# plot_exposure_distribution(sim_vectors, hs_code, country)
# hs_code = '2603'
# country = 'CN'
# plot_exposure_distribution(sim_vectors, hs_code, country)

In [0]:
import plotly.graph_objects as go


def plot_top5_comparison(sub_origins, sub_shares, hs_code, n=5, rank_by='shares'):
    label = HS_LABELS.get(hs_code, hs_code)

    if rank_by == 'shares':
        top = (
            sub_shares[sub_shares['hs_code'] == hs_code]
            .sort_values('global_share', ascending=False)
            .head(n)
        )
        df = top.merge(
            sub_origins[sub_origins['hs_code'] == hs_code][['country_code', 'average', 'certainty']],
            on='country_code',
            how='left'
        )
    elif rank_by == 'origins':
        top = (
            sub_origins[sub_origins['hs_code'] == hs_code]
            .sort_values('average', ascending=False)
            .head(n)
        )
        df = top.merge(
            sub_shares[sub_shares['hs_code'] == hs_code][['country_code', 'country_name', 'global_share']],
            on='country_code',
            how='left'
        )
    else:
        raise ValueError("rank_by must be 'shares' or 'origins'")

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df['country_code'],
        y=df['global_share'],
        name='Global net exports share',
        marker_color='#0080FF',  # electric blue
        customdata=df[['country_name']],
        hovertemplate="Country: %{customdata[0]}<br>Global Net Exports Share: %{y:.1%}<extra></extra>",
    ))

    fig.add_trace(go.Bar(
        x=df['country_code'],
        y=df['average'],
        name='Network exposure estimate',
        marker_color='#FF7A1A',  # electric orange
        customdata=df[['country_name', 'certainty']],
        hovertemplate=(
            "Country: %{customdata[0]}<br>"
            "Origins estimate: %{y:.1%}<br>"
            "Confidence: %{customdata[1]:.1%}"
            "<extra></extra>"
        ),
    ))

    rank_label = 'global net exports share' if rank_by == 'shares' else 'network origins estimate'
    fig.update_layout(
        barmode='group',
        height=550,
        width=850,
        title=dict(
            text=f"{label} ({hs_code}): network origins vs global net exports — top {n} by {rank_label}",
            x=0.5,
            xanchor='center',
        ),
        xaxis=dict(title='Country'),
        yaxis=dict(title='Share', tickformat='.0%'),
        legend_title_text='Source',
    )

    fig.show()


In [0]:
import pandas as pd

def prepare_data_for_sankey(sim_edges, start_country, hs_chain):

    start_hs_code = str(hs_chain[0])
    root_rows = pd.DataFrame([
        {
            'target_hs': 'root',
            'target_country': start_country,
            'target_terminal': 'production',
            'source_hs': start_hs_code,
            'source_country': start_country,
            'source_terminal': 'supply',
            'weight': 1.0,
            'replicate': i,
        }
        for i in sim_edges['replicate'].unique()
    ])
    flows = pd.concat([sim_edges, root_rows], ignore_index=True)
    flow_skip = pd.merge(flows, flows, how = 'left', left_on = ['source_hs', 'source_country', 'source_terminal', 'replicate'], right_on=['target_hs', 'target_country', 'target_terminal', 'replicate'])
    flow_skip['weight'] = np.minimum(flow_skip['weight_x'],flow_skip['weight_y'])
    flow_skip =flow_skip.drop(['weight_x', 'weight_y', 'source_hs_x', 'source_country_x', 'source_terminal_x', 'target_hs_y', 'target_country_y', 'target_terminal_y'], axis=1)
    flow_skip = flow_skip.rename(columns = {'target_hs_x':'target_hs', 'target_country_x':'target_country', 'target_terminal_x':'target_terminal', 'source_hs_y':'source_hs', 'source_country_y':'source_country', 'source_terminal_y':'source_terminal'})
    flow_skip = flow_skip[(flow_skip['target_terminal']=='production')&(flow_skip['source_terminal']=='production')]
    group_cols = [c for c in flow_skip.columns if c not in ['weight', 'replicate']]
    average_flow = flow_skip.groupby(group_cols).agg({'weight': 'mean'}).reset_index()
    variation_flow = flow_skip.groupby(group_cols).agg({'weight': 'std'}).reset_index()
    flow_results =pd.merge(average_flow, variation_flow, on = ['source_hs', 'source_country', 'source_terminal', 'target_hs', 'target_country', 'target_terminal']).rename(columns={'weight_x': 'average', 'weight_y': 'standard_deviation'}).fillna(0)

    """
    flow_skip: production->production edges (target_hs/target_country/target_terminal,
               source_hs/source_country/source_terminal, average), including a 'root'
               target_hs row for the starting country.
    start_country: country of final demand
    hs_chain: hs codes downstream -> upstream, e.g. [8703, 8501, 7408, 7403, 2603]

    Returns:
        columns: list of dicts, country -> cumulative (min-composed) weight.
                 columns[0] = {start_country: 1.0}
                 columns[k] (k>=1) = hs_chain[k-1] split across producing countries
        links: list of (col_from, country_from, col_to, country_to, value)
    """
    columns = [{start_country: 1.0}]
    links = []
    current_hs = 'root'

    for i, hs in enumerate(hs_chain):
        hs = str(hs)
        prev_col = columns[-1]
        edges = flow_results[(flow_results.target_hs == current_hs) & (flow_results.source_hs == hs)]

        new_col = {}
        for c_prev, prev_val in prev_col.items():
            rows = edges[edges.target_country == c_prev]
            for _, r in rows.iterrows():
                val = r.average
                new_col[r.source_country] = max(new_col.get(r.source_country, 0), val)
                links.append((i, c_prev, i + 1, r.source_country, val))

        columns.append(new_col)
        current_hs = hs

    return columns, links

def plot_root_sankey(columns, hs_chain, links, start_country, min_value=0.0, hs_labels=None):
    import plotly.graph_objects as go
    hs_labels = hs_labels or {}
 
    links = [l for l in links if l[4] >= min_value]
    used_nodes = {(l[0], l[1]) for l in links} | {(l[2], l[3]) for l in links}
    # always keep the root/start node even if nothing downstream survives the threshold
    used_nodes.add((0, start_country))
    columns = [{c: v for c, v in col.items() if (i, c) in used_nodes} for i, col in enumerate(columns)]
 
    import colorsys
    def make_palette(n):
        colors = []
        golden_ratio_conjugate = 0.618033988749895
        h = 0.05
        for i in range(n):
            h = (h + golden_ratio_conjugate) % 1
            s = 0.65
            v = 0.92 if i % 2 == 0 else 0.80
            r, g, b = colorsys.hsv_to_rgb(h, s, v)
            colors.append('#%02x%02x%02x' % (int(r * 255), int(g * 255), int(b * 255)))
        return colors
    palette = make_palette(40)
    country_colors = {}
    def color_for(c):
        if c not in country_colors:
            country_colors[c] = palette[len(country_colors) % len(palette)]
        return country_colors[c]
    def rgba(hexcolor, a=0.45):
        h = hexcolor.lstrip('#')
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f'rgba({r},{g},{b},{a})'
 
    col_labels = [f"{start_country} demand"] + [str(h) for h in hs_chain]
    n_cols = len(columns)
    node_labels, node_colors, node_index, node_x, node_y = [], [], {}, [], []
    col_x = []
    for i, col in enumerate(columns):
        ordered = sorted(col.items(), key=lambda kv: kv[1], reverse=True)
        n = len(ordered)
        x = 0.03 if n_cols == 1 else min(0.97, max(0.03, 0.03 + 0.94 * i / (n_cols - 1)))
        col_x.append(x if n > 0 else None)
        total = sum(v for _, v in ordered)
        cum = 0.0
        for country, v in ordered:
            node_index[(i, country)] = len(node_labels)
            node_labels.append(country)
            node_colors.append(color_for(country))
            node_x.append(x)
            if n == 1:
                node_y.append(0.5)
            else:
                y = (cum + v / 2) / total if total > 0 else 0.5
                node_y.append(0.02 + 0.96 * y)
                cum += v
 
    src, tgt, val, lcol = [], [], [], []
    for i_from, c_from, i_to, c_to, v in links:
        src.append(node_index[(i_from, c_from)])
        tgt.append(node_index[(i_to, c_to)])
        val.append(v)
        lcol.append(rgba(color_for(c_to)))
 
    incoming = {}
    for i_from, c_from, i_to, c_to, v in links:
        incoming[(i_to, c_to)] = incoming.get((i_to, c_to), 0.0) + v
 
    node_hover = []
    for (i, country), idx in sorted(node_index.items(), key=lambda kv: kv[1]):
        if i == 0:
            total_in = columns[i][country]
        else:
            total_in = incoming.get((i, country), 0.0)
        node_hover.append(f"{country} ({col_labels[i]}): {total_in:.1%} incoming")
 
    fig = go.Figure(go.Sankey(
        arrangement='fixed',
        node=dict(label=node_labels, color=node_colors, pad=25, thickness=16,
                  line=dict(color='white', width=0.5), x=node_x, y=node_y,
                  customdata=node_hover, hovertemplate='%{customdata}<extra></extra>'),
        link=dict(source=src, target=tgt, value=val, color=lcol, hoverinfo='skip')
    ))
    max_col_size = max((len(col) for col in columns), default=1)
    fig.update_layout(

        height=max(550, 70 * max_col_size),
        margin=dict(t=130, b=30, l=60, r=70)
    )
    for i, x in enumerate(col_x):
        if x is None:
            continue
        fig.add_annotation(
            x=x, y=1.09, xref='paper', yref='paper',
            text=f"<b>{col_labels[i]}</b>", showarrow=False,
            font=dict(size=13), xanchor='center'
        )
        desc = hs_labels.get(hs_chain[i - 1]) or hs_labels.get(str(hs_chain[i - 1])) if i > 0 else None
        if desc:
            fig.add_annotation(
                x=x, y=1.03, xref='paper', yref='paper',
                text=desc, showarrow=False,
                font=dict(size=10, color='gray'), xanchor='center'
            )
    return fig

In [0]:
# New sankey with variability shown
import pandas as pd
import numpy as np

def prepare_data_for_uncertainty_sankey(sim_edges, start_country, hs_chain):

    start_hs_code = str(hs_chain[0])
    root_rows = pd.DataFrame([
        {
            'target_hs': 'root',
            'target_country': start_country,
            'target_terminal': 'production',
            'source_hs': start_hs_code,
            'source_country': start_country,
            'source_terminal': 'supply',
            'weight': 1.0,
            'replicate': i,
        }
        for i in sim_edges['replicate'].unique()
    ])
    flows = pd.concat([sim_edges, root_rows], ignore_index=True)
    flow_skip = pd.merge(flows, flows, how='left',
                          left_on=['source_hs', 'source_country', 'source_terminal', 'replicate'],
                          right_on=['target_hs', 'target_country', 'target_terminal', 'replicate'])
    flow_skip['weight'] = np.minimum(flow_skip['weight_x'], flow_skip['weight_y'])
    flow_skip = flow_skip.drop(['weight_x', 'weight_y', 'source_hs_x', 'source_country_x',
                                 'source_terminal_x', 'target_hs_y', 'target_country_y', 'target_terminal_y'], axis=1)
    flow_skip = flow_skip.rename(columns={'target_hs_x': 'target_hs', 'target_country_x': 'target_country',
                                           'target_terminal_x': 'target_terminal', 'source_hs_y': 'source_hs',
                                           'source_country_y': 'source_country', 'source_terminal_y': 'source_terminal'})
    flow_skip = flow_skip[(flow_skip['target_terminal'] == 'production') & (flow_skip['source_terminal'] == 'production')]
    group_cols = [c for c in flow_skip.columns if c not in ['weight', 'replicate']]
    average_flow = flow_skip.groupby(group_cols).agg({'weight': 'mean'}).reset_index()
    variation_flow = flow_skip.groupby(group_cols).agg({'weight': 'std'}).reset_index()
    flow_results = pd.merge(
        average_flow, variation_flow,
        on=['source_hs', 'source_country', 'source_terminal', 'target_hs', 'target_country', 'target_terminal']
    ).rename(columns={'weight_x': 'average', 'weight_y': 'standard_deviation'}).fillna(0)

    columns = [{start_country: 1.0}]
    links = []  # now (i_from, c_from, i_to, c_to, value, std)
    current_hs = 'root'

    for i, hs in enumerate(hs_chain):
        hs = str(hs)
        prev_col = columns[-1]
        edges = flow_results[(flow_results.target_hs == current_hs) & (flow_results.source_hs == hs)]

        new_col = {}
        for c_prev, prev_val in prev_col.items():
            rows = edges[edges.target_country == c_prev]
            for _, r in rows.iterrows():
                val = r.average
                std = r.standard_deviation
                new_col[r.source_country] = new_col.get(r.source_country, 0) + val
                links.append((i, c_prev, i + 1, r.source_country, val, std))

        columns.append(new_col)
        current_hs = hs

    return columns, links

import pandas as pd
import numpy as np
def plot_root_uncertainty_sankey(columns, hs_chain, links, start_country, min_value=0.0, hs_labels=None):
    import plotly.graph_objects as go
    import colorsys
    hs_labels = hs_labels or {}

    links = [l for l in links if l[4] >= min_value]
    used_nodes = {(l[0], l[1]) for l in links} | {(l[2], l[3]) for l in links}
    used_nodes.add((0, start_country))
    columns = [{c: v for c, v in col.items() if (i, c) in used_nodes} for i, col in enumerate(columns)]

    NODE_BLUE = '#2c5f8a'

    def relative_std(value, std, eps=1e-9):
        return std / (value + eps)

    cvs = np.array([relative_std(v, s) for (_, _, _, _, v, s) in links])
    if len(cvs) > 0 and cvs.max() > cvs.min():
        lo, hi = np.percentile(cvs, [5, 95])
        if hi <= lo:
            lo, hi = cvs.min(), cvs.max()
    else:
        lo, hi = 0.0, 1.0

    DARK_HSV = (0.58, 0.85, 0.55)
    LIGHT_HSV = (0.58, 0.15, 0.97)
    DARK_A, LIGHT_A = 0.75, 0.20

    def blue_for_cv(cv):
        t = 0.0 if hi <= lo else np.clip((cv - lo) / (hi - lo), 0, 1)
        h = DARK_HSV[0] + (LIGHT_HSV[0] - DARK_HSV[0]) * t
        s = DARK_HSV[1] + (LIGHT_HSV[1] - DARK_HSV[1]) * t
        v = DARK_HSV[2] + (LIGHT_HSV[2] - DARK_HSV[2]) * t
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        alpha = DARK_A + (LIGHT_A - DARK_A) * t
        return f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha:.3f})'

    col_labels = [f"{start_country} demand"] + [str(h) for h in hs_chain]
    n_cols = len(columns)
    node_labels, node_colors, node_index, node_x, node_y = [], [], {}, [], []
    col_x = []
    for i, col in enumerate(columns):
        ordered = sorted(col.items(), key=lambda kv: kv[1], reverse=True)
        n = len(ordered)
        x = 0.03 if n_cols == 1 else min(0.97, max(0.03, 0.03 + 0.94 * i / (n_cols - 1)))
        col_x.append(x if n > 0 else None)
        total = sum(v for _, v in ordered)
        cum = 0.0
        for country, v in ordered:
            node_index[(i, country)] = len(node_labels)
            node_labels.append(country)
            node_colors.append(NODE_BLUE)
            node_x.append(x)
            if n == 1:
                node_y.append(0.5)
            else:
                y = (cum + v / 2) / total if total > 0 else 0.5
                node_y.append(0.02 + 0.96 * y)
                cum += v

    src, tgt, val, lcol, lhover = [], [], [], [], []
    for i_from, c_from, i_to, c_to, v, std in links:
        src.append(node_index[(i_from, c_from)])
        tgt.append(node_index[(i_to, c_to)])
        val.append(v)
        cv = relative_std(v, std)
        lcol.append(blue_for_cv(cv))
        lhover.append(f"{c_from} → {c_to}<br>share: {v:.1%}<br>uncertainty (std/mean): {cv:.1%}")

    # Aggregate incoming value and a value-weighted average CV per node
    incoming = {}
    incoming_weighted_cv = {}
    for i_from, c_from, i_to, c_to, v, std in links:
        key = (i_to, c_to)
        incoming[key] = incoming.get(key, 0.0) + v
        incoming_weighted_cv[key] = incoming_weighted_cv.get(key, 0.0) + v * relative_std(v, std)

    node_hover = []
    for (i, country), idx in sorted(node_index.items(), key=lambda kv: kv[1]):
        if i == 0:
            total_in = columns[i][country]
            node_hover.append(f"{country} ({col_labels[i]}): {total_in:.1%} incoming")
        else:
            key = (i, country)
            total_in = incoming.get(key, 0.0)
            avg_cv = incoming_weighted_cv.get(key, 0.0) / total_in if total_in > 0 else 0.0
            node_hover.append(
                f"{country} ({col_labels[i]}): {total_in:.1%} incoming<br>"
                f"uncertainty (std/mean): {avg_cv:.1%}"
            )

    fig = go.Figure(go.Sankey(
        arrangement='fixed',
        textfont=dict(size=14, family='Arial Black'),
        node=dict(label=node_labels, color=node_colors, pad=25, thickness=16,
                  line=dict(color='white', width=0.5), x=node_x, y=node_y,
                  customdata=node_hover, hovertemplate='%{customdata}<extra></extra>'),
        link=dict(source=src, target=tgt, value=val, color=lcol,
                  customdata=lhover, hovertemplate='%{customdata}<extra></extra>')
    ))
    max_col_size = max((len(col) for col in columns), default=1)
    fig.update_layout(
        height=max(550, 70 * max_col_size),
        margin=dict(t=130, b=30, l=60, r=70)
    )
    for i, x in enumerate(col_x):
        if x is None:
            continue
        fig.add_annotation(
            x=x, y=1.09, xref='paper', yref='paper',
            text=f"<b>{col_labels[i]}</b>", showarrow=False,
            font=dict(size=13), xanchor='center'
        )
        desc = hs_labels.get(hs_chain[i - 1]) or hs_labels.get(str(hs_chain[i - 1])) if i > 0 else None
        if desc:
            fig.add_annotation(
                x=x, y=1.03, xref='paper', yref='paper',
                text=desc, showarrow=False,
                font=dict(size=10, color='gray'), xanchor='center'
            )
    return fig

# Cocao network

In [0]:
product_map = product_map_cocoa
hs_codes = hs_chain_cocoa   
HS_LABELS = {    # --- 4-digit headings ---
    "1801": "Cocoa beans",
    "1802": "Cocoa shells",
    "1803": "Cocoa paste",
    "1804": "Cocoa butter",
    "1805": "Cocoa powder",
    "1806": "Chocolate products"
    }


In [0]:
# Execute supply network trace
start_hs = '1806'
start_country = 'AU'
replicates = 30
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:

plot_top5_comparison(origins, global_shares, "1801", n=10, rank_by='origins')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, 'AU', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Steel in car supply network

In [0]:
import pandas as pd
import numpy as np

In [0]:
product_map = product_map_steel

HS_LABELS = {
    "8703": "Motor vehicles",
    "8707": "Vehicle bodies",
    "8501": "Electric motors",
    "8708": "Vehicle parts/accessories",
    "4011": "New pneumatic tyres",
    "7308": "Iron/steel structures",
    "7616": "Aluminum articles",
    "7408": "Copper wire",
    "7409": "Copper plates/sheet/strip",
    "7217": "Steel wire",
    "4001": "Natural rubber",
    "8542": "Integrated circuits",
    "7207": "Semi-finished steel",
    "7208": "Hot-rolled flat steel",
    "7209": "Cold-rolled flat steel",
    "7210": "Coated/plated flat steel",
    "7225": "Flat-rolled alloy steel",
    "7226": "Other flat-rolled alloy steel",
    "7601": "Raw aluminum",
    "7403": "Refined copper",
    "2601": "Iron ore",
    "2606": "Bauxite",
    "2603": "Copper ore",
}

hs_codes = hs_chain_steel
start_hs_code = '8703'

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '8703'
replicates = 20
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
display(origins)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
plot_top5_comparison(origins, global_shares, "2601", n=10, rank_by='origins')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, 'AU', min_value=0.005, hs_labels=HS_LABELS)
fig.show()

# Copper in car supply network

In [0]:
product_map = product_map_cu
hs_codes = hs_chain_cu

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '8703'
replicates = 20
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, 'AU', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Aluminum in car network

In [0]:
product_map = product_map_al
hs_codes = hs_chain_al

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '8703'
replicates = 20
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, 'AU', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Cobalt in EV Batteries


In [0]:
product_map_cobalt = pd.DataFrame([
    ("8703", "8507", 1.00),   # vehicle <- electric battery
    ("8507", "8105", 1.00),   # batteries <- cobalt mattes/intermediate products/refined cobalt
    ("8105", "2605", 1.00),   # cobalt mattes/refined <- cobalt ores and concentrates
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_cobalt = ["8703","8507", "8105", "2605"]
HS_LABELS.update({
    "8507": "Batteries",
    "8105": "Cobalt mattes/refined cobalt",
    "2605": "Cobalt ore",
})

product_map = product_map_cobalt
hs_codes = hs_chain_cobalt

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '8703'
replicates = 20
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, start_country, hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, start_country, min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Nickel in EV batteries

In [0]:
product_map_nickel = pd.DataFrame([
    ("8703", "8507", 1.00),   # vehicle <- electric battery
    ("8507", "7501", 1.00),   # batteries <- nickel mattes/nickel oxide sinters/intermediate products
    ("7501", "2604", 1.00),   # nickel mattes/intermediate <- nickel ores and concentrates
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_nickel = ["8703", "8507", "7501", "2604"]
HS_LABELS.update({
    "8507": "Batteries",
    "7501": "Nickel mattes/intermediate",
    "2604": "Nickel ore",
})

product_map = product_map_nickel
hs_codes = hs_chain_nickel

In [0]:

product_map = product_map_nickel
hs_codes = hs_chain_nickel

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '8703'
replicates = 20
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, start_country, hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, start_country, min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Copper supply network

In [0]:
import pandas as pd

# Copper chain (4-digit HS)
product_map_copper = pd.DataFrame([
    ("7408", "7403", 1.00),   # copper wire <- refined copper
    ("7403", "2603", 1.00),   # refined copper <- copper ore/concentrate
], columns=["target_hs_code", "source_hs_code", "weight"])
product_map = product_map_copper
hs_chain_copper = ["7408", "7403", "2603"]

HS_LABELS.update({
    "2603": "Copper ores and concentrates",
    "7403": "Refined copper and copper alloys, unwrought",
    "7408": "Copper wire",
})

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '7408'
replicates = 30
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_chain_copper)
fig = plot_root_uncertainty_sankey(columns, hs_chain_copper, links, 'AU', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Scrap Copper Chain

In [0]:
import pandas as pd

# Copper chain (4-digit HS)
product_map_copper = pd.DataFrame([
    ("7408", "7404", 1.00),   # copper wire <- scrap copper
], columns=["target_hs_code", "source_hs_code", "weight"])
product_map = product_map_copper
hs_chain_copper = ["7408", "7404"]

HS_LABELS.update({
    "7404": "Scrap copper",
    "7408": "Copper wire",
})

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = '7408'
replicates = 30
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'AU', hs_chain_copper)
fig = plot_root_uncertainty_sankey(columns, hs_chain_copper, links, 'AU', min_value=0.005, hs_labels=HS_LABELS)
fig.show()

# Textiles

In [0]:
# Cotton -> textiles -> apparel chain (4-digit HS)
product_map_textiles = pd.DataFrame([
    ("6109", "5208", 1.00),   # t-shirts/apparel <- woven cotton fabric
    ("5208", "5205", 1.00),   # woven cotton fabric <- cotton yarn
    ("5205", "5201", 1.00),   # cotton yarn <- raw cotton (not carded/combed)
], columns=["target_hs_code", "source_hs_code", "weight"])

hs_chain_textiles = ["6109", "5208", "5205", "5201"]

HS_LABELS.update({
    "5201": "Cotton, not carded or combed (raw)",
    "5205": "Cotton yarn (single, not for retail)",
    "5208": "Woven cotton fabric (light weight)",
    "6109": "T-shirts, singlets, knitted or crocheted",
})
product_map = product_map_textiles
hs_codes = hs_chain_textiles

In [0]:
# Execute supply network trace
start_country = 'US'
start_hs = hs_codes[0]
replicates = 30
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
plot_top5_comparison(origins, global_shares, "5201", n=10, rank_by='origins')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, 'US', hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, 'US', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

# Integrated Circuits

In [0]:
HS_LABELS.update({
    "8542": "Integrated circuits",     # already in your dict — kept here for reference
    "3818": "Doped semiconductor wafers",
    "2804": "Polysilicon",
})
product_map = product_map_semiconductor
hs_codes = hs_chain_semiconductor

In [0]:
# Execute supply network trace
start_country = 'AU'
start_hs = hs_codes[0]
replicates = 30
sim_vectors, sim_edges, origins =  network_trace_iterator(product_map, start_country, start_hs, replicates)
global_shares = load_global_net_exports(hs_codes)

In [0]:
plot_prioritisation_matrix(origins, .02, measure='median')

In [0]:
columns, links = prepare_data_for_uncertainty_sankey(sim_edges, start_country, hs_codes)
fig = plot_root_uncertainty_sankey(columns, hs_codes, links, start_country, min_value=0.01, hs_labels=HS_LABELS)
fig.show()

In [0]:
plot_top5_comparison(origins, global_shares, "8542", n=10, rank_by='origins')
plot_top5_comparison(origins, global_shares, "3818", n=10, rank_by='origins')
plot_top5_comparison(origins, global_shares, "2804", n=10, rank_by='origins')

In [0]:
ISO2_TO_ISO3 = {
    "AD":"AND","AE":"ARE","AF":"AFG","AG":"ATG","AI":"AIA","AL":"ALB","AM":"ARM","AO":"AGO",
    "AQ":"ATA","AR":"ARG","AS":"ASM","AT":"AUT","AU":"AUS","AW":"ABW","AX":"ALA","AZ":"AZE",
    "BA":"BIH","BB":"BRB","BD":"BGD","BE":"BEL","BF":"BFA","BG":"BGR","BH":"BHR","BI":"BDI",
    "BJ":"BEN","BL":"BLM","BM":"BMU","BN":"BRN","BO":"BOL","BQ":"BES","BR":"BRA","BS":"BHS",
    "BT":"BTN","BV":"BVT","BW":"BWA","BY":"BLR","BZ":"BLZ","CA":"CAN","CC":"CCK","CD":"COD",
    "CF":"CAF","CG":"COG","CH":"CHE","CI":"CIV","CK":"COK","CL":"CHL","CM":"CMR","CN":"CHN",
    "CO":"COL","CR":"CRI","CU":"CUB","CV":"CPV","CW":"CUW","CX":"CXR","CY":"CYP","CZ":"CZE",
    "DE":"DEU","DJ":"DJI","DK":"DNK","DM":"DMA","DO":"DOM","DZ":"DZA","EC":"ECU","EE":"EST",
    "EG":"EGY","EH":"ESH","ER":"ERI","ES":"ESP","ET":"ETH","FI":"FIN","FJ":"FJI","FK":"FLK",
    "FM":"FSM","FO":"FRO","FR":"FRA","GA":"GAB","GB":"GBR","GD":"GRD","GE":"GEO","GF":"GUF",
    "GG":"GGY","GH":"GHA","GI":"GIB","GL":"GRL","GM":"GMB","GN":"GIN","GP":"GLP","GQ":"GNQ",
    "GR":"GRC","GS":"SGS","GT":"GTM","GU":"GUM","GW":"GNB","GY":"GUY","HK":"HKG","HM":"HMD",
    "HN":"HND","HR":"HRV","HT":"HTI","HU":"HUN","ID":"IDN","IE":"IRL","IL":"ISR","IM":"IMN",
    "IN":"IND","IO":"IOT","IQ":"IRQ","IR":"IRN","IS":"ISL","IT":"ITA","JE":"JEY","JM":"JAM",
    "JO":"JOR","JP":"JPN","KE":"KEN","KG":"KGZ","KH":"KHM","KI":"KIR","KM":"COM","KN":"KNA",
    "KP":"PRK","KR":"KOR","KW":"KWT","KY":"CYM","KZ":"KAZ","LA":"LAO","LB":"LBN","LC":"LCA",
    "LI":"LIE","LK":"LKA","LR":"LBR","LS":"LSO","LT":"LTU","LU":"LUX","LV":"LVA","LY":"LBY",
    "MA":"MAR","MC":"MCO","MD":"MDA","ME":"MNE","MF":"MAF","MG":"MDG","MH":"MHL","MK":"MKD",
    "ML":"MLI","MM":"MMR","MN":"MNG","MO":"MAC","MP":"MNP","MQ":"MTQ","MR":"MRT","MS":"MSR",
    "MT":"MLT","MU":"MUS","MV":"MDV","MW":"MWI","MX":"MEX","MY":"MYS","MZ":"MOZ","NA":"NAM",
    "NC":"NCL","NE":"NER","NF":"NFK","NG":"NGA","NI":"NIC","NL":"NLD","NO":"NOR","NP":"NPL",
    "NR":"NRU","NU":"NIU","NZ":"NZL","OM":"OMN","PA":"PAN","PE":"PER","PF":"PYF","PG":"PNG",
    "PH":"PHL","PK":"PAK","PL":"POL","PM":"SPM","PN":"PCN","PR":"PRI","PS":"PSE","PT":"PRT",
    "PW":"PLW","PY":"PRY","QA":"QAT","RE":"REU","RO":"ROU","RS":"SRB","RU":"RUS","RW":"RWA",
    "SA":"SAU","SB":"SLB","SC":"SYC","SD":"SDN","SE":"SWE","SG":"SGP","SH":"SHN","SI":"SVN",
    "SJ":"SJM","SK":"SVK","SL":"SLE","SM":"SMR","SN":"SEN","SO":"SOM","SR":"SUR","SS":"SSD",
    "ST":"STP","SV":"SLV","SX":"SXM","SY":"SYR","SZ":"SWZ","TC":"TCA","TD":"TCD","TF":"ATF",
    "TG":"TGO","TH":"THA","TJ":"TJK","TK":"TKL","TL":"TLS","TM":"TKM","TN":"TUN","TO":"TON",
    "TR":"TUR","TT":"TTO","TV":"TUV","TW":"TWN","TZ":"TZA","UA":"UKR","UG":"UGA","UM":"UMI",
    "US":"USA","UY":"URY","UZ":"UZB","VA":"VAT","VC":"VCT","VE":"VEN","VG":"VGB","VI":"VIR",
    "VN":"VNM","VU":"VUT","WF":"WLF","WS":"WSM","YE":"YEM","YT":"MYT","ZA":"ZAF","ZM":"ZMB",
    "ZW":"ZWE"
}

plot_dependency_map(origins, '8703')
plot_dependency_map(origins, '4011')
plot_dependency_map(origins, '4001')
# plot_dependency_map(origins, '7403')
# plot_dependency_map(origins, '2603')

In [0]:
import pandas as pd

def prepare_data_for_sankey(sim_edges, start_country, hs_chain):

    start_hs_code = str(hs_chain[0])
    root_rows = pd.DataFrame([
        {
            'target_hs': 'root',
            'target_country': start_country,
            'target_terminal': 'production',
            'source_hs': start_hs_code,
            'source_country': start_country,
            'source_terminal': 'supply',
            'weight': 1.0,
            'replicate': i,
        }
        for i in sim_edges['replicate'].unique()
    ])
    flows = pd.concat([sim_edges, root_rows], ignore_index=True)
    flow_skip = pd.merge(flows, flows, how = 'left', left_on = ['source_hs', 'source_country', 'source_terminal', 'replicate'], right_on=['target_hs', 'target_country', 'target_terminal', 'replicate'])
    flow_skip['weight'] = np.minimum(flow_skip['weight_x'],flow_skip['weight_y'])
    flow_skip =flow_skip.drop(['weight_x', 'weight_y', 'source_hs_x', 'source_country_x', 'source_terminal_x', 'target_hs_y', 'target_country_y', 'target_terminal_y'], axis=1)
    flow_skip = flow_skip.rename(columns = {'target_hs_x':'target_hs', 'target_country_x':'target_country', 'target_terminal_x':'target_terminal', 'source_hs_y':'source_hs', 'source_country_y':'source_country', 'source_terminal_y':'source_terminal'})
    flow_skip = flow_skip[(flow_skip['target_terminal']=='production')&(flow_skip['source_terminal']=='production')]
    group_cols = [c for c in flow_skip.columns if c not in ['weight', 'replicate']]
    average_flow = flow_skip.groupby(group_cols).agg({'weight': 'mean'}).reset_index()
    variation_flow = flow_skip.groupby(group_cols).agg({'weight': 'std'}).reset_index()
    flow_results =pd.merge(average_flow, variation_flow, on = ['source_hs', 'source_country', 'source_terminal', 'target_hs', 'target_country', 'target_terminal']).rename(columns={'weight_x': 'average', 'weight_y': 'standard_deviation'}).fillna(0)

    """
    flow_skip: production->production edges (target_hs/target_country/target_terminal,
               source_hs/source_country/source_terminal, average), including a 'root'
               target_hs row for the starting country.
    start_country: country of final demand
    hs_chain: hs codes downstream -> upstream, e.g. [8703, 8501, 7408, 7403, 2603]

    Returns:
        columns: list of dicts, country -> cumulative (min-composed) weight.
                 columns[0] = {start_country: 1.0}
                 columns[k] (k>=1) = hs_chain[k-1] split across producing countries
        links: list of (col_from, country_from, col_to, country_to, value)
    """
    columns = [{start_country: 1.0}]
    links = []
    current_hs = 'root'

    for i, hs in enumerate(hs_chain):
        hs = str(hs)
        prev_col = columns[-1]
        edges = flow_results[(flow_results.target_hs == current_hs) & (flow_results.source_hs == hs)]

        new_col = {}
        for c_prev, prev_val in prev_col.items():
            rows = edges[edges.target_country == c_prev]
            for _, r in rows.iterrows():
                val = r.average
                new_col[r.source_country] = max(new_col.get(r.source_country, 0), val)
                links.append((i, c_prev, i + 1, r.source_country, val))

        columns.append(new_col)
        current_hs = hs

    return columns, links

def plot_root_sankey(columns, hs_chain, links, start_country, min_value=0.0, hs_labels=None):
    import plotly.graph_objects as go
    hs_labels = hs_labels or {}
 
    links = [l for l in links if l[4] >= min_value]
    used_nodes = {(l[0], l[1]) for l in links} | {(l[2], l[3]) for l in links}
    # always keep the root/start node even if nothing downstream survives the threshold
    used_nodes.add((0, start_country))
    columns = [{c: v for c, v in col.items() if (i, c) in used_nodes} for i, col in enumerate(columns)]
 
    import colorsys
    def make_palette(n):
        colors = []
        golden_ratio_conjugate = 0.618033988749895
        h = 0.05
        for i in range(n):
            h = (h + golden_ratio_conjugate) % 1
            s = 0.65
            v = 0.92 if i % 2 == 0 else 0.80
            r, g, b = colorsys.hsv_to_rgb(h, s, v)
            colors.append('#%02x%02x%02x' % (int(r * 255), int(g * 255), int(b * 255)))
        return colors
    palette = make_palette(40)
    country_colors = {}
    def color_for(c):
        if c not in country_colors:
            country_colors[c] = palette[len(country_colors) % len(palette)]
        return country_colors[c]
    def rgba(hexcolor, a=0.45):
        h = hexcolor.lstrip('#')
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f'rgba({r},{g},{b},{a})'
 
    col_labels = [f"{start_country} demand"] + [str(h) for h in hs_chain]
    n_cols = len(columns)
    node_labels, node_colors, node_index, node_x, node_y = [], [], {}, [], []
    col_x = []
    for i, col in enumerate(columns):
        ordered = sorted(col.items(), key=lambda kv: kv[1], reverse=True)
        n = len(ordered)
        x = 0.03 if n_cols == 1 else min(0.97, max(0.03, 0.03 + 0.94 * i / (n_cols - 1)))
        col_x.append(x if n > 0 else None)
        total = sum(v for _, v in ordered)
        cum = 0.0
        for country, v in ordered:
            node_index[(i, country)] = len(node_labels)
            node_labels.append(country)
            node_colors.append(color_for(country))
            node_x.append(x)
            if n == 1:
                node_y.append(0.5)
            else:
                y = (cum + v / 2) / total if total > 0 else 0.5
                node_y.append(0.02 + 0.96 * y)
                cum += v
 
    src, tgt, val, lcol = [], [], [], []
    for i_from, c_from, i_to, c_to, v in links:
        src.append(node_index[(i_from, c_from)])
        tgt.append(node_index[(i_to, c_to)])
        val.append(v)
        lcol.append(rgba(color_for(c_to)))
 
    incoming = {}
    for i_from, c_from, i_to, c_to, v in links:
        incoming[(i_to, c_to)] = incoming.get((i_to, c_to), 0.0) + v
 
    node_hover = []
    for (i, country), idx in sorted(node_index.items(), key=lambda kv: kv[1]):
        if i == 0:
            total_in = columns[i][country]
        else:
            total_in = incoming.get((i, country), 0.0)
        node_hover.append(f"{country} ({col_labels[i]}): {total_in:.1%} incoming")
 
    fig = go.Figure(go.Sankey(
        arrangement='fixed',
        node=dict(label=node_labels, color=node_colors, pad=25, thickness=16,
                  line=dict(color='white', width=0.5), x=node_x, y=node_y,
                  customdata=node_hover, hovertemplate='%{customdata}<extra></extra>'),
        link=dict(source=src, target=tgt, value=val, color=lcol, hoverinfo='skip')
    ))
    max_col_size = max((len(col) for col in columns), default=1)
    fig.update_layout(

        height=max(550, 70 * max_col_size),
        margin=dict(t=130, b=30, l=60, r=70)
    )
    for i, x in enumerate(col_x):
        if x is None:
            continue
        fig.add_annotation(
            x=x, y=1.09, xref='paper', yref='paper',
            text=f"<b>{col_labels[i]}</b>", showarrow=False,
            font=dict(size=13), xanchor='center'
        )
        desc = hs_labels.get(hs_chain[i - 1]) or hs_labels.get(str(hs_chain[i - 1])) if i > 0 else None
        if desc:
            fig.add_annotation(
                x=x, y=1.03, xref='paper', yref='paper',
                text=desc, showarrow=False,
                font=dict(size=10, color='gray'), xanchor='center'
            )
    return fig

In [0]:
HS_LABELS = {
    "1806": "Chocolate products",
    "1803": "Cocoa paste",
    "1801": "Cocoa beans",
}

In [0]:
hs_codes = ['8703', '8707', '7308', '7207', '2601']
columns, links = prepare_data_for_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_sankey(columns, hs_codes, links, 'AU', min_value=0.01, hs_labels=HS_LABELS)
fig.show()

In [0]:
import pandas as pd

def apply_scenario_override(sim_edges, target_hs, source_hs, target_country,
                             forced_source_country,
                             target_terminal='production', source_terminal='production'):
    """
    Forces all inbound weight for a specific (source_hs -> target_hs, target_country) edge
    to be attributed entirely to `forced_source_country`, preserving the total weight
    (so downstream conservation still holds), for every replicate.
    """
    edges = sim_edges.copy()

    mask = (
        (edges['target_hs'] == target_hs) &
        (edges['source_hs'] == source_hs) &
        (edges['target_country'] == target_country) &
        (edges['target_terminal'] == target_terminal)
    )
    scoped = edges[mask]
    if scoped.empty:
        raise ValueError(
            f"No edges found for source_hs={source_hs} -> target_hs={target_hs}, "
            f"target_country={target_country}."
        )

    # total inbound weight per replicate, before the override — this gets preserved
    totals = scoped.groupby('replicate')['weight'].sum()

    # drop the existing (multi-country) edges for this slice...
    edges = edges[~mask]

    # ...and replace with a single edge attributing 100% of that weight to forced_source_country
    new_rows = pd.DataFrame({
        'source_hs': source_hs,
        'source_country': forced_source_country,
        'source_terminal': source_terminal,
        'target_hs': target_hs,
        'target_country': target_country,
        'target_terminal': target_terminal,
        'weight': totals.values,
        'replicate': totals.index,
    })

    return pd.concat([edges, new_rows], ignore_index=True)


# Example: force 7308 (steel structures) -> 8707 (body) for Japan to be 100% domestic
scenario_edges = apply_scenario_override(
    sim_edges,
    target_hs='8707',
    source_hs='7308',
    target_country='JP',
    forced_source_country='JP',
)

columns, links = prepare_data_for_sankey(scenario_edges, start_country='JP', hs_chain=hs_codes)
fig = plot_root_sankey(columns, hs_codes, links, start_country='JP', hs_labels=HS_LABELS)
fig.show()

In [0]:
scenario_edges

In [0]:
hs_codes = ['8703', '4011', '7217', '2601']
columns, links = prepare_data_for_sankey(sim_edges, 'AU', hs_codes)
fig = plot_root_sankey(columns, hs_codes, links, 'AU', min_value=0.001, hs_labels=HS_LABELS)
fig.show()

In [0]:
plot_prioritisation_matrix(origins, .02)

In [0]:
hs_code = '1806'
country = 'CA'
plot_exposure_distribution(sim_vectors, hs_code, country)
hs_code = '1803'
country = 'CA'
plot_exposure_distribution(sim_vectors, hs_code, country)
hs_code = '1801'
country = 'CA'
plot_exposure_distribution(sim_vectors, hs_code, country)

In [0]:
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    # Pull only the records you need directly from the cluster database
    query = f"""
        SELECT * 
        FROM silver_country_imports 
        WHERE hs_code IN ({hs_codes_str})
    """
    imports = spark.sql(query).toPandas()

In [0]:
imports

In [0]:
    hs_codes_str = ", ".join(f"'{code}'" for code in hs_codes)
    query = f"""
        SELECT * 
        FROM silver_country_hs_code_hyperparamers_2
        WHERE hs_code IN ({hs_codes_str})
        AND a IS NOT NULL
    """

    imports = spark.sql(query).toPandas()

In [0]:
imports['Net Exports']= imports['exports']-imports['imports']

In [0]:
imports[imports['hs_code']=='2603'].sort_values('Net Exports', ascending = False).head(30)

In [0]:
imports[imports['country_code']=='CD']

In [0]:

    root_rows = pd.DataFrame([
        {
            'target_hs': 'root',
            'target_country': start_country,
            'target_terminal': 'production',
            'source_hs': start_hs,
            'source_country': start_country,
            'source_terminal': 'supply',
            'weight': 1.0,
            'replicate': i,
        }
        for i in sim_edges['replicate'].unique()
    ])
    flows = pd.concat([sim_edges, root_rows], ignore_index=True)
    flow_skip = pd.merge(flows, flows, how = 'left', left_on = ['source_hs', 'source_country', 'source_terminal', 'replicate'], right_on=['target_hs', 'target_country', 'target_terminal', 'replicate'])
    flow_skip['weight'] = np.minimum(flow_skip['weight_x'],flow_skip['weight_y'])
    flow_skip =flow_skip.drop(['weight_x', 'weight_y', 'source_hs_x', 'source_country_x', 'source_terminal_x', 'target_hs_y', 'target_country_y', 'target_terminal_y'], axis=1)
    flow_skip = flow_skip.rename(columns = {'target_hs_x':'target_hs', 'target_country_x':'target_country', 'target_terminal_x':'target_terminal', 'source_hs_y':'source_hs', 'source_country_y':'source_country', 'source_terminal_y':'source_terminal'})
    flow_skip = flow_skip[(flow_skip['target_terminal']=='production')&(flow_skip['source_terminal']=='production')]
    group_cols = [c for c in flow_skip.columns if c not in ['weight', 'replicate']]
    average_flow = flow_skip.groupby(group_cols).agg({'weight': 'mean'}).reset_index()
    variation_flow = flow_skip.groupby(group_cols).agg({'weight': 'std'}).reset_index()
    flow_results =pd.merge(average_flow, variation_flow, on = ['source_hs', 'source_country', 'source_terminal', 'target_hs', 'target_country', 'target_terminal']).rename(columns={'weight_x': 'average', 'weight_y': 'standard_deviation'}).fillna(0)

In [0]:
flow_results[(flow_results['target_country']=='CN') & (flow_results['target_hs']=='7308') & (flow_results['target_terminal']=='production')]

In [0]:
flow_skip[flow_skip['target_country']=='JP'].sort_values('average', ascending = False).head(20)

In [0]:
def plot_sankey(
    edges_df,
    hs_codes,
    hs_labels=None,
    min_value=1e-4,
    palette=None,
    width=1200,
    height=700,
    vertical_padding=0.02,
):
    import pandas as pd
    import plotly.graph_objects as go

    # =========================================================
    # Helper
    # =========================================================

    def hex_to_rgba(hex_color, alpha=1.0):
        hex_color = hex_color.lstrip("#")

        if len(hex_color) == 3:
            hex_color = "".join(c * 2 for c in hex_color)

        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)

        return f"rgba({r}, {g}, {b}, {alpha})"

    hs_labels = hs_labels or {}

    # =========================================================
    # Prepare data
    # =========================================================

    df = edges_df.copy()

    hs_set = set(hs_codes)

    df = df[
        df["source_hs"].isin(hs_set)
        & df["target_hs"].isin(hs_set)
    ].copy()

    if df.empty:
        print("No edges found for requested HS codes.")
        return

    # =========================================================
    # Contract production nodes
    # =========================================================

    production_inputs = df[
        df["target_terminal"] == "production"
    ].copy()

    production_outputs = df[
        df["source_terminal"] == "production"
    ].copy()

    production_input_lookup = {}

    for _, row in production_inputs.iterrows():

        key = (
            row["target_hs"],
            row["target_country"],
        )

        production_input_lookup.setdefault(
            key,
            []
        ).append(row)

    contracted_edges = []

    for _, output in production_outputs.iterrows():

        production_key = (
            output["source_hs"],
            output["source_country"],
        )

        inputs = production_input_lookup.get(
            production_key,
            []
        )

        for input_edge in inputs:

            contracted_edges.append({
                "source_hs": input_edge["source_hs"],
                "source_country": input_edge["source_country"],
                "source_terminal": "supply",

                "target_hs": output["target_hs"],
                "target_country": output["target_country"],
                "target_terminal": "supply",

                "weight": (
                    input_edge["weight"]
                    * output["weight"]
                ),
            })

    # =========================================================
    # Preserve direct supply → supply edges
    # =========================================================

    direct_supply_edges = df[
        (df["source_terminal"] == "supply")
        & (df["target_terminal"] == "supply")
    ].copy()

    for _, row in direct_supply_edges.iterrows():

        contracted_edges.append({
            "source_hs": row["source_hs"],
            "source_country": row["source_country"],
            "source_terminal": "supply",

            "target_hs": row["target_hs"],
            "target_country": row["target_country"],
            "target_terminal": "supply",

            "weight": row["weight"],
        })

    df = pd.DataFrame(contracted_edges)

    if df.empty:
        print(
            "No contracted edges found. "
            "Check your production/supply structure."
        )
        return

    # =========================================================
    # Aggregate duplicate edges
    # =========================================================

    df = (
        df.groupby(
            [
                "source_hs",
                "source_country",
                "source_terminal",
                "target_hs",
                "target_country",
                "target_terminal",
            ],
            as_index=False,
        )["weight"]
        .sum()
    )

    # =========================================================
    # Minimum weight filter
    # =========================================================

    df = df[
        df["weight"] >= min_value
    ].copy()

    if df.empty:
        print(
            f"No edges left at min_value={min_value} "
            "— try lowering it."
        )
        return

    # =========================================================
    # HS codes
    # =========================================================

    present_hs = [
        hs
        for hs in hs_codes
        if (
            hs in set(df["source_hs"])
            or hs in set(df["target_hs"])
        )
    ]

    hs_rank = {
        hs: i
        for i, hs in enumerate(present_hs)
    }

    # =========================================================
    # Country colours
    # =========================================================

    country_totals = pd.concat([
        df.groupby("source_country")["weight"].sum(),
        df.groupby("target_country")["weight"].sum(),
    ]).groupby(level=0).sum().sort_values(
        ascending=False
    )

    all_countries = country_totals.index.tolist()

    country_order = {
        country: i
        for i, country in enumerate(all_countries)
    }

    default_palette = [
        "#378ADD",
        "#1D9E75",
        "#D85A30",
        "#D4537E",
        "#7F77DD",
        "#BA7517",
        "#639922",
        "#888780",
        "#E24B4A",
        "#5DCAA5",
    ]

    palette = palette or default_palette

    country_color = {
        country: palette[i % len(palette)]
        for i, country in enumerate(all_countries)
    }

    # =========================================================
    # Node keys
    # =========================================================

    df["source_key"] = list(zip(
        df["source_hs"],
        df["source_country"],
    ))

    df["target_key"] = list(zip(
        df["target_hs"],
        df["target_country"],
    ))

    all_keys = (
        pd.Index(df["source_key"])
        .append(
            pd.Index(df["target_key"])
        )
        .unique()
    )

    key_to_index = {
        key: i
        for i, key in enumerate(all_keys)
    }

    # =========================================================
    # X positions
    # =========================================================

    n_hs = len(present_hs)

    if n_hs == 1:

        hs_x = {
            present_hs[0]: 0.5
        }

    else:

        hs_x = {
            hs: (
                0.04
                + 0.92
                * hs_rank[hs]
                / (n_hs - 1)
            )
            for hs in present_hs
        }

    # =========================================================
    # Group nodes by HS
    # =========================================================

    hs_members = {}

    for key in all_keys:

        hs = key[0]

        hs_members.setdefault(
            hs,
            []
        ).append(key)

    # =========================================================
    # Node positions
    #
    # Countries are given explicit gaps.
    # =========================================================

    node_x = [0.5] * len(all_keys)
    node_y = [0.5] * len(all_keys)

    node_color = [
        "#888780"
        for _ in all_keys
    ]

    for hs, members in hs_members.items():

        x = hs_x[hs]

        members_sorted = sorted(
            members,
            key=lambda key: country_order.get(
                key[1],
                999
            )
        )

        n = len(members_sorted)

        # -----------------------------------------------------
        # Available vertical space
        # -----------------------------------------------------

        top = 0.04
        bottom = 0.96

        available = bottom - top

        # Desired spacing between node centres
        #
        # This creates more separation when there are
        # relatively few countries.
        # -----------------------------------------------------

        if n == 1:

            y_positions = [0.5]

        else:

            spacing = min(
                (available / (n - 1)),
                0.075
            )

            total_height = (
                spacing * (n - 1)
            )

            start = (
                0.5
                - total_height / 2
            )

            y_positions = [
                start + j * spacing
                for j in range(n)
            ]

        # -----------------------------------------------------
        # Assign positions
        # -----------------------------------------------------

        for j, key in enumerate(
            members_sorted
        ):

            idx = key_to_index[key]

            node_x[idx] = x
            node_y[idx] = y_positions[j]

            node_color[idx] = (
                country_color.get(
                    key[1],
                    "#888780"
                )
            )

    # =========================================================
    # Node labels
    # =========================================================

    node_labels = [
        key[1]
        for key in all_keys
    ]

    # =========================================================
    # Link colours
    # =========================================================

    link_colors = [
        hex_to_rgba(
            country_color.get(
                country,
                "#888780"
            ),
            0.45,
        )
        for country in df["source_country"]
    ]

    # =========================================================
    # Link hover
    # =========================================================

    link_customdata = [
        (
            f"<b>{src[1]}</b> · {src[0]}"
            f" → "
            f"<b>{tgt[1]}</b> · {tgt[0]}"
            f"<br>"
            f"weight: {weight:.6f}"
        )
        for src, tgt, weight in zip(
            df["source_key"],
            df["target_key"],
            df["weight"],
        )
    ]

    # =========================================================
    # Sankey
    # =========================================================

    fig = go.Figure(
        go.Sankey(

            arrangement="fixed",

            node=dict(
                x=node_x,
                y=node_y,

                color=node_color,

                label=node_labels,

                # Smaller nodes, but with more separation
                pad=10,
                thickness=7,

                line=dict(
                    color="black",
                    width=0.2,
                ),

                hovertemplate=(
                    "<b>%{label}</b>"
                    "<extra></extra>"
                ),
            ),

            link=dict(
                source=df["source_key"].map(
                    key_to_index
                ),

                target=df["target_key"].map(
                    key_to_index
                ),

                value=df["weight"],

                color=link_colors,

                customdata=link_customdata,

                hovertemplate=(
                    "%{customdata}"
                    "<extra></extra>"
                ),
            ),
        )
    )

    # =========================================================
    # HS labels
    # =========================================================

    annotations = []

    for hs in present_hs:

        annotations.append(
            dict(
                x=hs_x[hs],
                y=1.04,

                xref="paper",
                yref="paper",

                text=(
                    f"<b>{hs}</b>"
                    f"<br>"
                    f"<span style='font-size:8px'>"
                    f"{hs_labels.get(hs, '')}"
                    f"</span>"
                ),

                showarrow=False,

                xanchor="center",
                yanchor="bottom",

                font=dict(
                    size=9
                ),
            )
        )

    # =========================================================
    # Layout
    # =========================================================

    fig.update_layout(

        width=width,
        height=height,

        margin=dict(
            l=10,
            r=20,
            t=65,
            b=10,
        ),

        font=dict(
            size=8
        ),

        annotations=annotations,
    )

    # =========================================================
    # Display
    # =========================================================

    fig.show()

    # =========================================================
    # Legend
    # =========================================================

    print("\nHS code legend:")

    for hs in present_hs:

        print(
            f"  {hs}: "
            f"{hs_labels.get(hs, '(no label)')}"
        )

    return fig

In [0]:
edges_df[edges_df['source_terminal']=='production']


In [0]:
hs_codes = ['8703', '8707', '7308', '7207', '2601']

In [0]:
edges_df.sort_values('weight', ascending = False)

In [0]:
# Execute supply network trace
fig = plot_sankey(edges_df, hs_codes=["START"] + list(hs_codes), hs_labels={**HS_LABELS, "START": "Start"}, min_value=1e-4)


In [0]:
pip install -U kaleido

In [0]:
import os

os.path.exists("/tmp/sankey.html")


In [0]:
ISO2_TO_ISO3 = {
    "AD":"AND","AE":"ARE","AF":"AFG","AG":"ATG","AI":"AIA","AL":"ALB","AM":"ARM","AO":"AGO",
    "AQ":"ATA","AR":"ARG","AS":"ASM","AT":"AUT","AU":"AUS","AW":"ABW","AX":"ALA","AZ":"AZE",
    "BA":"BIH","BB":"BRB","BD":"BGD","BE":"BEL","BF":"BFA","BG":"BGR","BH":"BHR","BI":"BDI",
    "BJ":"BEN","BL":"BLM","BM":"BMU","BN":"BRN","BO":"BOL","BQ":"BES","BR":"BRA","BS":"BHS",
    "BT":"BTN","BV":"BVT","BW":"BWA","BY":"BLR","BZ":"BLZ","CA":"CAN","CC":"CCK","CD":"COD",
    "CF":"CAF","CG":"COG","CH":"CHE","CI":"CIV","CK":"COK","CL":"CHL","CM":"CMR","CN":"CHN",
    "CO":"COL","CR":"CRI","CU":"CUB","CV":"CPV","CW":"CUW","CX":"CXR","CY":"CYP","CZ":"CZE",
    "DE":"DEU","DJ":"DJI","DK":"DNK","DM":"DMA","DO":"DOM","DZ":"DZA","EC":"ECU","EE":"EST",
    "EG":"EGY","EH":"ESH","ER":"ERI","ES":"ESP","ET":"ETH","FI":"FIN","FJ":"FJI","FK":"FLK",
    "FM":"FSM","FO":"FRO","FR":"FRA","GA":"GAB","GB":"GBR","GD":"GRD","GE":"GEO","GF":"GUF",
    "GG":"GGY","GH":"GHA","GI":"GIB","GL":"GRL","GM":"GMB","GN":"GIN","GP":"GLP","GQ":"GNQ",
    "GR":"GRC","GS":"SGS","GT":"GTM","GU":"GUM","GW":"GNB","GY":"GUY","HK":"HKG","HM":"HMD",
    "HN":"HND","HR":"HRV","HT":"HTI","HU":"HUN","ID":"IDN","IE":"IRL","IL":"ISR","IM":"IMN",
    "IN":"IND","IO":"IOT","IQ":"IRQ","IR":"IRN","IS":"ISL","IT":"ITA","JE":"JEY","JM":"JAM",
    "JO":"JOR","JP":"JPN","KE":"KEN","KG":"KGZ","KH":"KHM","KI":"KIR","KM":"COM","KN":"KNA",
    "KP":"PRK","KR":"KOR","KW":"KWT","KY":"CYM","KZ":"KAZ","LA":"LAO","LB":"LBN","LC":"LCA",
    "LI":"LIE","LK":"LKA","LR":"LBR","LS":"LSO","LT":"LTU","LU":"LUX","LV":"LVA","LY":"LBY",
    "MA":"MAR","MC":"MCO","MD":"MDA","ME":"MNE","MF":"MAF","MG":"MDG","MH":"MHL","MK":"MKD",
    "ML":"MLI","MM":"MMR","MN":"MNG","MO":"MAC","MP":"MNP","MQ":"MTQ","MR":"MRT","MS":"MSR",
    "MT":"MLT","MU":"MUS","MV":"MDV","MW":"MWI","MX":"MEX","MY":"MYS","MZ":"MOZ","NA":"NAM",
    "NC":"NCL","NE":"NER","NF":"NFK","NG":"NGA","NI":"NIC","NL":"NLD","NO":"NOR","NP":"NPL",
    "NR":"NRU","NU":"NIU","NZ":"NZL","OM":"OMN","PA":"PAN","PE":"PER","PF":"PYF","PG":"PNG",
    "PH":"PHL","PK":"PAK","PL":"POL","PM":"SPM","PN":"PCN","PR":"PRI","PS":"PSE","PT":"PRT",
    "PW":"PLW","PY":"PRY","QA":"QAT","RE":"REU","RO":"ROU","RS":"SRB","RU":"RUS","RW":"RWA",
    "SA":"SAU","SB":"SLB","SC":"SYC","SD":"SDN","SE":"SWE","SG":"SGP","SH":"SHN","SI":"SVN",
    "SJ":"SJM","SK":"SVK","SL":"SLE","SM":"SMR","SN":"SEN","SO":"SOM","SR":"SUR","SS":"SSD",
    "ST":"STP","SV":"SLV","SX":"SXM","SY":"SYR","SZ":"SWZ","TC":"TCA","TD":"TCD","TF":"ATF",
    "TG":"TGO","TH":"THA","TJ":"TJK","TK":"TKL","TL":"TLS","TM":"TKM","TN":"TUN","TO":"TON",
    "TR":"TUR","TT":"TTO","TV":"TUV","TW":"TWN","TZ":"TZA","UA":"UKR","UG":"UGA","UM":"UMI",
    "US":"USA","UY":"URY","UZ":"UZB","VA":"VAT","VC":"VCT","VE":"VEN","VG":"VGB","VI":"VIR",
    "VN":"VNM","VU":"VUT","WF":"WLF","WS":"WSM","YE":"YEM","YT":"MYT","ZA":"ZAF","ZM":"ZMB",
    "ZW":"ZWE"
}

plot_dependency_map(vector_df, '8703')
plot_dependency_map(vector_df, '8707')
plot_dependency_map(vector_df, '7308')
plot_dependency_map(vector_df, '7207')
plot_dependency_map(vector_df, '2601')

In [0]:
def extract_vector(vector, index_to_tuple):

    series = pd.Series(vector.toarray().ravel())

    long_df = pd.DataFrame(
        [index_to_tuple[i] for i in series.index],
        columns=["hs_code", "country_code", "country_terminal"]
    )
    long_df["weight"] = series.values

    # drop zeros if it's sparse and you only care about nonzero entries
    long_df = long_df[long_df["value"] != 0]
    return long_df


In [0]:
# Display results
result = cumulative_vector.tocoo()

result_df = pd.DataFrame({
    'node': result.col,
    'weight': result.data
}).sort_values('weight', ascending = False)
result_df['node_tuple'] = result_df['node'].map(index_to_tuple)
result_df['hs_code'],  result_df['country_code'], result_df['country_terminal']  = zip(*result_df['node_tuple'])
display(result_df)


In [0]:
result_df.hs_code.unique()

In [0]:
transition_rows['tuple']

In [0]:
def generate_country_sources(alphas, relevant_hs_codes):
    domestic_share = generate_domestic_production_share(alphas, relevant_hs_codes)
    hs_codes_str = ", ".join(f"'{code}'" for code in relevant_hs_codes)

    # Pull only the records you need directly from the cluster database
    query = f"""
        SELECT * 
        FROM silver_country_imports 
        WHERE hs_code IN ({hs_codes_str})
    """

    imports = spark.sql(query).toPandas()
    sources = imports.merge(domestic_share, how = 'left', left_on = ['hs_code', 'source_country_code'], right_on = ['hs_code', 'country_code']).drop(columns='country_code_y').rename(columns={'country_code_x': 'country_code'}).copy()
    sources = sources[['country_code', 'country', 'source_country_code', 'source_country_name', 'hs_code','import_share', 'domestic_share']].copy()
    return sources

In [0]:
def single_node_trace(start_country, hs_code, country_sources):
    source = country_sources[(country_sources['country_code']==start_country) & (country_sources['hs_code'] ==hs_code)].copy()
    source['domestic_share'] = source['domestic_share'].fillna(0)
    source['final_source'] = source['import_share']*source['domestic_share']
    source['residual_source'] = source['import_share']*(1-source['domestic_share'])
    final_source = source[['hs_code', 'source_country_code', 'source_country_name', 'final_source']].copy()
    residual_source = source[['hs_code','source_country_code', 'source_country_name', 'residual_source']].copy()
    return final_source, residual_source

In [0]:

def network_traversal(alphas, hs_code, start_country, country_sources, consider_domestic = True):

    network = pd.DataFrame(columns=[
    'hs_code',
    'source_country_code',
    'source_country_name',
    'final_source',
    'target_country'
    ])

    residual_total = pd.DataFrame(columns=[
    'hs_code',
    'source_country_code',
    'source_country_name',
    'residual_source'
    ])

    current_country = start_country

    domestic_production = 0

    if consider_domestic:
        alpha_row = alphas[
            (alphas['country_code'] == start_country) &
            (alphas['hs_code'] == hs_code)
        ]
        print(alpha_row)

        if not alpha_row.empty:
            domestic_production = alpha_row['alpha'].iloc[0]

    modifier = 1 - domestic_production
    residual_sum = modifier
    while residual_sum > 0.01:
        print(f'performing single node trace for {current_country}')
        final, residual = single_node_trace(current_country, hs_code, country_sources)
        if final.empty and residual.empty:
            break
        final = final.copy()
        residual = residual.copy()
        final['final_source'] = modifier * final['final_source']
        residual['residual_source'] = modifier * residual['residual_source']
        final['target_country'] = current_country
        network = pd.concat([network, final], ignore_index=True)
        residual_total = pd.concat([residual_total, residual], ignore_index=True)
        residual_total = residual_total.groupby(
            ['hs_code', 'source_country_code', 'source_country_name'], as_index=False
            ).sum().sort_values('residual_source', ascending = False)
        residual_sum =residual_total['residual_source'].sum()
        if residual_total.empty:
            break
        # print(f'residual sum is now {residual_sum}')
        current_country = residual_total.iloc[0]['source_country_code']
        modifier = residual_total.iloc[0]['residual_source']
        residual_total = residual_total.iloc[1:]


    final_contribution = network.groupby(['hs_code', 'source_country_code', 'source_country_name'], as_index = False)['final_source'].sum()
    if consider_domestic:
        final_contribution = pd.concat(
            [
                final_contribution,
                pd.DataFrame([{
                    'hs_code': hs_code,
                    'source_country_code': start_country,
                    'source_country_name': 'Domestic',
                    'final_source': domestic_production
                }])
            ],
            ignore_index=True
        )
    return final_contribution, network, residual_sum



In [0]:
relevant_hs_codes = {
    '8703',
    '8707',
}
hs_code = '8703'
start_country = 'AU'
alphas = generate_alpha_dictionary(relevant_hs_codes)
country_sources = generate_country_sources(alphas, relevant_hs_codes)
final_contribution, network, residual_sum = network_traversal(alphas, hs_code, start_country, country_sources)

In [0]:
final_contribution.sort_values('final_source', ascending = False)


In [0]:
# def replicate_network_traversal(start_country, hs_code, replicates=10, consider_domestic = True):
#     relevant_hs_codes = {hs_code}
#     contributions = pd.DataFrame()
#     networks = pd.DataFrame()

#     for i in range(replicates):
#         print(f'running replicate {i}')
#         alphas = generate_alpha_dictionary(relevant_hs_codes)
#         country_sources = generate_country_sources(alphas, relevant_hs_codes)
#         final_contribution, network, residual_sum = network_traversal(hs_code, start_country, country_sources, consider_domestic)
#         final_contribution['replicate'] = i
#         network['replicate'] = i
#         contributions = pd.concat([contributions, final_contribution])
#         networks = pd.concat([networks, network])

#     contributions['target_country'] = start_country
#     return contributions, networks




In [0]:
# input product map

product_map = pd.DataFrame([
    ("8703", "8707", 0.55),
    ("8703", "8501", 0.20),
    ("8703", "8507", 0.25),

    ("8707", "7308", 0.10),
    ("8707", "7616", 0.30),
    ("8707", "4011", 0.20),

    ("8501", "7408", 0.40),
    ("8501", "8482", 0.30),
    ("8501", "8542", 0.20),

    ("8507", "8542", 0.10),
    ("8507", "7616", 0.20),
    ("8507", "7408", 0.10),
], columns=["parent_hs", "child_hs", "weight"])




In [0]:

# product map functions 

def get_parents(hs_code, product_map):
    return set(product_map[product_map["child_hs"] == hs_code]["parent_hs"])

def get_children(hs_code, product_map):
    return set(product_map[product_map["parent_hs"] == hs_code]["child_hs"])

def get_next_node(product_map, completed_nodes):

    if not completed_nodes:
        root_nodes = set(product_map['parent_hs']) - set(product_map['child_hs'])
        return next(iter(root_nodes))
    
    sub_map = product_map[(product_map['parent_hs'].isin(completed_nodes)) & ~(product_map['child_hs'].isin(completed_nodes))].copy()

    if sub_map.empty:
        return None
    
    return sub_map.iloc[0]['child_hs']


In [0]:
# inputs
root_country = 'AU'
root_hs_code = '8703'
replicates = 2

# HS code tracing
completed_nodes =set()
all_nodes = set(product_map['parent_hs']).union(set(product_map['child_hs']))

# calculate tables
alphas = generate_alpha_dictionary(relevant_hs_codes)
country_sources = generate_country_sources(alphas, relevant_hs_codes)

# Do root node
contributions, networks, residual_sum = network_traversal(alphas, hs_code, start_country, country_sources)
completed_nodes.add(root_hs_code)


In [0]:
contributions.sort_values('final_source', ascending = False).head(30)

In [0]:
current_hs_code = get_next_node(product_map, completed_nodes)
current_parents = get_parents(current_hs_code, product_map)
print(f"Current HS code: {current_hs_code}")

In [0]:
total_contributions = filtered_contributions.groupby(['source_country_code', 'source_country_name', 'target_country'], as_index = False)['final_source'].sum().sort_values('final_source', ascending = False)

display(total_contributions)


In [0]:
next_countries_dic = dict(zip(total_contributions['source_country_code'], total_contributions['final_source']))
next_countries_dic

In [0]:
# Sample motor car example starting in Australia

# inputs
root_country = 'AU'
root_hs_code = '8703'

# HS code tracing
completed_nodes =set()
all_nodes = set(product_map['parent_hs']).union(set(product_map['child_hs']))

# calculate tables
alphas = generate_alpha_dictionary(all_nodes)
country_sources = generate_country_sources(alphas, all_nodes)



# Do root node
contributions, networks, residual_sum = network_traversal(alphas, root_hs_code, root_country, country_sources)
contributions['target_country_code'] = root_country
completed_nodes.add(root_hs_code)

# Traverse product map
while len(completed_nodes)!= len(all_nodes):
    current_hs_code = get_next_node(product_map, completed_nodes)
    current_parents = get_parents(current_hs_code, product_map)
    print(f"Current HS code: {current_hs_code}")

    # Identify next countries and their weights

    # Filter to most important contributors
    # Better filtered contribution that ensure the sum of what is take exceend .9
    filtered_contributions = (
        contributions[
            (contributions['hs_code'].isin(current_parents))
            & (contributions['final_source'] > 0)
        ]
        .sort_values('final_source', ascending=False)
    )

    filtered_contributions['cumulative'] = filtered_contributions['final_source'].cumsum()

    # Keep rows up to (and including) the one that pushes the total over 0.9
    filtered_contributions = filtered_contributions[
        (filtered_contributions['cumulative'] <= 0.9)
        | (filtered_contributions['cumulative'].shift(fill_value=0) < 0.9)
    ]

    # Dealing with product map weights
    sub_product_map = product_map[(product_map['parent_hs'].isin(list(current_parents))) & (product_map['child_hs']==current_hs_code)]
    filtered_contributions = filtered_contributions.merge(sub_product_map, left_on = 'hs_code', right_on = 'parent_hs', how = 'left').sort_values('final_source',)
    filtered_contributions['final_source'] = filtered_contributions['final_source']*filtered_contributions['weight']

    filtered_contributions.sort_values('final_source', ascending = False)

    # Sum of HS code parents
    total_contributions = filtered_contributions.groupby(['source_country_code', 'source_country_name', 'target_country_code'], as_index = False)['final_source'].sum().sort_values('final_source', ascending = False)
    next_countries_dic = dict(zip(total_contributions['source_country_code'], total_contributions['final_source']))

    counter = 0
    for country, weight in next_countries_dic.items():
        print(f"Tracing HS code {current_hs_code} to {country} with weight {weight}")
        counter += 1
        new_contributions, new_networks, residual_sum = network_traversal(alphas, current_hs_code, country, country_sources)
        new_contributions['target_country_code'] = country
        new_contributions['final_source'] = weight * new_contributions['final_source'] 
        new_networks['final_source'] = weight * new_networks['final_source']

        contributions = pd.concat([contributions, new_contributions])
        networks = pd.concat([networks, new_networks])

    completed_nodes.add(current_hs_code)


spark_contributions = spark.createDataFrame(contributions)
spark_contributions.write.mode("overwrite").saveAsTable("gold_motor_vehicle_02_contributions")
spark_networks = spark.createDataFrame(networks)
spark_networks.write.mode("overwrite").saveAsTable("gold_motor_vehicle_02_networks")

In [0]:
spark_contributions.write.mode("overwrite").saveAsTable("gold_motor_vehicle_02_contributions")
spark_networks = spark.createDataFrame(networks)
spark_networks.write.mode("overwrite").saveAsTable("gold_motor_vehicle_02_networks")

In [0]:
contributions[contributions['hs_code']=='7408'].sort_values('final_source', ascending = False).head(10)

In [0]:

product_map

In [0]:
trade = spark.table("gold_motor_vehicle_01_contributions")

In [0]:
detrade = trade.groupby(['hs_code', 'source_country_code', 'source_country_name', 'target_country']).agg({'final_source': 'mean'}).orderBy('mean(final_source)', ascending=False)


In [0]:
from pyspark.sql import functions as F

display(
    detrade
        .filter(F.col("hs_code") == "7308")
        .agg(F.sum("mean(final_source)").alias("total"))
)

In [0]:

new_contributions.sort_values('final_source', ascending = False)

In [0]:
new_networks

In [0]:
source_country

In [0]:
contributions.sort_values('final_source', ascending = False).head(20)

In [0]:
product_map

In [0]:
def build_parent_map(bom):
    """
    Build a mapping of each HS code to its parent HS codes.

    Example output:
    {
        "8703": set(),
        "8707": {"8703"},
        "7308": {"8707"},
        "7408": {"8501", "8507"},
        ...
    }
    """
    parents = {}

    def recurse(node, parent=None):
        for hs_code, children in node.items():
            parents.setdefault(hs_code, set())

            if parent is not None:
                parents[hs_code].add(parent)

            recurse(children, hs_code)

    recurse(bom)

    return parents

In [0]:
parents = build_parent_map(motor_vehicle)

In [0]:
parents

In [0]:
def get_next_ready_node(parents, done=None):
    done = set(done or [])

    for hs_code, required_parents in parents.items():
        if hs_code not in done and required_parents <= done:
            return hs_code

    return None

In [0]:
done = None
code = get_next_ready_node(parents, done)

In [0]:
code

In [0]:
df = contributions[contributions['source_country_code']=='JP']

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean styling theme
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

# Plot the histogram for Japan's final_source column
# kde=True adds a smooth trend line; bins=5 works best for small datasets
sns.histplot(
    data=df, 
    x="final_source", 
    bins=10, 
    kde=True, 
    color="royalblue", 
    edgecolor="black"
)

# Add clear descriptive labels and vertical reference line for the mean
mean_val = df["final_source"].mean()
plt.axvline(mean_val, color="crimson", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.3f}")

# Layout formatting
plt.title("Distribution of Simulation Replicates for X (Passenger Cars - 8703)", fontsize=13, fontweight='bold')
plt.xlabel("Final Source Value", fontsize=11)
plt.ylabel("Frequency (Count)", fontsize=11)
plt.legend()

# Display the chart cleanly in your notebook output
plt.show()

In [0]:
final_contribution = final_total.groupby(['hs_code', 'source_country_code', 'source_country_name'], as_index = False)['final_source'].sum()

In [0]:
final

In [0]:
final['target_country'] = current_country
final